# 🎯 Options Pricing Engine — Black-Scholes + Monte Carlo
### Quantitative Finance Project | Intermediate Level
**Skills demonstrated:** Stochastic calculus · Option pricing · Greek letters · Volatility surface · Real market data

---
**Author:** [Ton nom] | **Date:** 2026 | **Tools:** Python, Yahoo Finance API

> *This project replicates the core pricing tools used on options desks at Goldman Sachs, BNP Paribas, and JPMorgan.*

In [ ]:
# ============================================================
# CELL 1 — INSTALLATION DES LIBRAIRIES
# ============================================================
!pip install yfinance scipy numpy matplotlib seaborn pandas plotly -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Style professionnel
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Toutes les librairies chargées avec succès')
print('📊 Options Pricing Engine prêt')

## 📐 Partie 1 — Modèle Black-Scholes

Le modèle de Black-Scholes (1973) est la fondation du pricing d'options. Il suppose que le prix de l'actif sous-jacent suit un **mouvement brownien géométrique** :

$$dS = \mu S \, dt + \sigma S \, dW_t$$

**Prix d'un Call :** $C = S_0 N(d_1) - K e^{-rT} N(d_2)$

**Prix d'un Put :** $P = K e^{-rT} N(-d_2) - S_0 N(-d_1)$

Où $d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$ et $d_2 = d_1 - \sigma\sqrt{T}$

In [ ]:
# ============================================================
# CELL 2 — IMPLÉMENTATION BLACK-SCHOLES COMPLÈTE
# ============================================================

class BlackScholes:
    """
    Moteur de pricing Black-Scholes complet.
    Calcule : prix, Greeks (Delta, Gamma, Vega, Theta, Rho)
    """

    def __init__(self, S, K, T, r, sigma, option_type='call'):
        """
        Parameters:
        -----------
        S     : Prix spot du sous-jacent (ex: 150.0)
        K     : Strike price (ex: 155.0)
        T     : Temps avant expiration en années (ex: 0.5 = 6 mois)
        r     : Taux sans risque annualisé (ex: 0.05 = 5%)
        sigma : Volatilité implicite annualisée (ex: 0.20 = 20%)
        option_type : 'call' ou 'put'
        """
        self.S = S
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
        self.option_type = option_type.lower()

    def _d1_d2(self):
        """Calcule les paramètres d1 et d2 de Black-Scholes"""
        d1 = (np.log(self.S / self.K) + (self.r + 0.5 * self.sigma**2) * self.T) / \
             (self.sigma * np.sqrt(self.T))
        d2 = d1 - self.sigma * np.sqrt(self.T)
        return d1, d2

    def price(self):
        """Prix théorique Black-Scholes"""
        d1, d2 = self._d1_d2()
        if self.option_type == 'call':
            price = (self.S * norm.cdf(d1) -
                     self.K * np.exp(-self.r * self.T) * norm.cdf(d2))
        else:
            price = (self.K * np.exp(-self.r * self.T) * norm.cdf(-d2) -
                     self.S * norm.cdf(-d1))
        return price

    def delta(self):
        """Delta : sensibilité du prix au sous-jacent"""
        d1, _ = self._d1_d2()
        if self.option_type == 'call':
            return norm.cdf(d1)
        return norm.cdf(d1) - 1

    def gamma(self):
        """Gamma : sensibilité du delta au sous-jacent (courbure)"""
        d1, _ = self._d1_d2()
        return norm.pdf(d1) / (self.S * self.sigma * np.sqrt(self.T))

    def vega(self):
        """Vega : sensibilité au prix à la volatilité (pour 1% de vol)"""
        d1, _ = self._d1_d2()
        return self.S * norm.pdf(d1) * np.sqrt(self.T) * 0.01

    def theta(self):
        """Theta : décroissance temporelle par jour calendaire"""
        d1, d2 = self._d1_d2()
        term1 = -(self.S * norm.pdf(d1) * self.sigma) / (2 * np.sqrt(self.T))
        if self.option_type == 'call':
            theta = term1 - self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        else:
            theta = term1 + self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(-d2)
        return theta / 365  # Par jour

    def rho(self):
        """Rho : sensibilité aux taux d'intérêt (pour 1% de taux)"""
        _, d2 = self._d1_d2()
        if self.option_type == 'call':
            return self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(d2) * 0.01
        return -self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(-d2) * 0.01

    def summary(self):
        """Rapport complet prix + Greeks"""
        print('=' * 55)
        print(f'  BLACK-SCHOLES PRICER — {self.option_type.upper()}')
        print('=' * 55)
        print(f'  Paramètres   : S={self.S}, K={self.K}, T={self.T:.2f}y')
        print(f'                 r={self.r:.1%}, σ={self.sigma:.1%}')
        print('-' * 55)
        print(f'  Prix BS      : ${self.price():.4f}')
        print('-' * 55)
        print(f'  Δ Delta      : {self.delta():.4f}')
        print(f'  Γ Gamma      : {self.gamma():.6f}')
        print(f'  ν Vega       : {self.vega():.4f}  (per 1% vol)')
        print(f'  Θ Theta      : {self.theta():.4f}  (per day)')
        print(f'  ρ Rho        : {self.rho():.4f}  (per 1% rate)')
        print('=' * 55)


# ── EXEMPLE : Apple Call option ──────────────────────────────
bs_call = BlackScholes(S=190, K=195, T=0.25, r=0.053, sigma=0.28, option_type='call')
bs_call.summary()

print()
bs_put = BlackScholes(S=190, K=195, T=0.25, r=0.053, sigma=0.28, option_type='put')
bs_put.summary()

## 🎲 Partie 2 — Simulation Monte Carlo

Monte Carlo simule des **milliers de trajectoires** du prix selon :

$$S_T = S_0 \cdot \exp\left[(r - \frac{\sigma^2}{2})T + \sigma\sqrt{T}\,Z\right], \quad Z \sim \mathcal{N}(0,1)$$

Le prix de l'option est la **valeur actualisée de l'espérance des payoffs** :

$$C \approx e^{-rT} \cdot \mathbb{E}[\max(S_T - K, 0)]$$

In [ ]:
# ============================================================
# CELL 3 — SIMULATION MONTE CARLO AVEC VISUALISATION
# ============================================================

def monte_carlo_pricer(S, K, T, r, sigma, option_type='call',
                        n_simulations=100_000, n_steps=252, seed=42):
    """
    Pricer Monte Carlo avec trajectoires complètes.

    Returns:
    --------
    price      : Prix estimé
    std_error  : Erreur standard (intervalle de confiance)
    paths      : Trajectoires simulées (pour visualisation)
    """
    np.random.seed(seed)
    dt = T / n_steps

    # Simulation des incréments browniens (anti-thetic variates pour réduction de variance)
    Z = np.random.standard_normal((n_simulations // 2, n_steps))
    Z = np.concatenate([Z, -Z])  # Anti-thetic variates

    # Construction des trajectoires
    drift = (r - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * Z
    log_returns = drift + diffusion

    paths = S * np.exp(np.cumsum(log_returns, axis=1))
    paths = np.hstack([np.full((n_simulations, 1), S), paths])

    # Calcul du payoff à expiration
    S_T = paths[:, -1]
    if option_type.lower() == 'call':
        payoffs = np.maximum(S_T - K, 0)
    else:
        payoffs = np.maximum(K - S_T, 0)

    # Actualisation
    price = np.exp(-r * T) * np.mean(payoffs)
    std_error = np.exp(-r * T) * np.std(payoffs) / np.sqrt(n_simulations)

    return price, std_error, paths


# ── Paramètres ───────────────────────────────────────────────
S, K, T, r, sigma = 190, 195, 0.25, 0.053, 0.28

mc_price, mc_se, paths = monte_carlo_pricer(S, K, T, r, sigma, 'call', n_simulations=50_000)
bs_price = BlackScholes(S, K, T, r, sigma, 'call').price()

print('=' * 50)
print('  COMPARAISON BLACK-SCHOLES vs MONTE CARLO')
print('=' * 50)
print(f'  BS Price   : ${bs_price:.4f}')
print(f'  MC Price   : ${mc_price:.4f} ± {1.96*mc_se:.4f} (95% CI)')
print(f'  Écart      : {abs(bs_price - mc_price):.4f} ({abs(bs_price - mc_price)/bs_price*100:.2f}%)')
print('=' * 50)


# ── Visualisation des trajectoires ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1 : Trajectoires Monte Carlo
ax1 = axes[0]
time_axis = np.linspace(0, T * 252, paths.shape[1])
n_display = 200
for i in range(n_display):
    color = '#2196F3' if paths[i, -1] > K else '#EF5350'
    ax1.plot(time_axis, paths[i], alpha=0.15, linewidth=0.5, color=color)

ax1.axhline(y=K, color='black', linestyle='--', linewidth=2, label=f'Strike K={K}')
ax1.axhline(y=S, color='gray', linestyle=':', linewidth=1.5, label=f'Spot S={S}')
ax1.set_xlabel('Jours', fontsize=12)
ax1.set_ylabel('Prix ($)', fontsize=12)
ax1.set_title(f'Monte Carlo — {n_display} trajectoires\n🔵 In-the-money  🔴 Out-of-the-money', fontsize=13)
ax1.legend(fontsize=11)

# Plot 2 : Distribution des prix finaux
ax2 = axes[1]
S_T_all = paths[:, -1]
itm_mask = S_T_all > K
ax2.hist(S_T_all[~itm_mask], bins=60, color='#EF5350', alpha=0.7, label=f'OTM ({(~itm_mask).sum()/len(S_T_all)*100:.1f}%)')
ax2.hist(S_T_all[itm_mask], bins=60, color='#2196F3', alpha=0.7, label=f'ITM ({itm_mask.sum()/len(S_T_all)*100:.1f}%)')
ax2.axvline(x=K, color='black', linestyle='--', linewidth=2, label=f'Strike K={K}')
ax2.axvline(x=np.mean(S_T_all), color='orange', linestyle='-', linewidth=2, label=f'Moyenne: ${np.mean(S_T_all):.2f}')
ax2.set_xlabel('Prix à expiration ($)', fontsize=12)
ax2.set_ylabel('Fréquence', fontsize=12)
ax2.set_title(f'Distribution des prix finaux\nMC Price = ${mc_price:.4f} | BS Price = ${bs_price:.4f}', fontsize=13)
ax2.legend(fontsize=11)

plt.suptitle('Monte Carlo Simulation — Options Pricing', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('monte_carlo_simulation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée : monte_carlo_simulation.png')

## 🔬 Partie 3 — Analyse Complète des Greeks

In [ ]:
# ============================================================
# CELL 4 — VISUALISATION DES GREEKS
# ============================================================

# Paramètres de base
S_base, K_base, T_base, r_base, sigma_base = 190, 195, 0.25, 0.053, 0.28

# Grille de prix spot
S_range = np.linspace(130, 260, 300)

# Calcul des Greeks pour calls et puts
greeks_data = {}
for opt in ['call', 'put']:
    greeks_data[opt] = {
        'price' : [BlackScholes(s, K_base, T_base, r_base, sigma_base, opt).price() for s in S_range],
        'delta' : [BlackScholes(s, K_base, T_base, r_base, sigma_base, opt).delta() for s in S_range],
        'gamma' : [BlackScholes(s, K_base, T_base, r_base, sigma_base, opt).gamma() for s in S_range],
        'vega'  : [BlackScholes(s, K_base, T_base, r_base, sigma_base, opt).vega() for s in S_range],
        'theta' : [BlackScholes(s, K_base, T_base, r_base, sigma_base, opt).theta() for s in S_range],
    }

# Couleurs
CALL_COLOR = '#1565C0'
PUT_COLOR  = '#C62828'

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
greeks_info = [
    ('price', 'Prix de l\'option ($)', 'Prix'),
    ('delta', 'Delta (Δ)', 'Delta'),
    ('gamma', 'Gamma (Γ)', 'Gamma'),
    ('vega',  'Vega (ν) — per 1% vol', 'Vega'),
    ('theta', 'Theta (Θ) — per day ($)', 'Theta'),
]

for idx, (greek, ylabel, title) in enumerate(greeks_info):
    ax = axes[idx]
    ax.plot(S_range, greeks_data['call'][greek], color=CALL_COLOR, linewidth=2.5, label='Call')
    ax.plot(S_range, greeks_data['put'][greek],  color=PUT_COLOR,  linewidth=2.5, label='Put', linestyle='--')
    ax.axvline(x=K_base, color='black', linestyle=':', linewidth=1.5, alpha=0.8, label=f'Strike K={K_base}')
    ax.axvline(x=S_base, color='gray',  linestyle=':', linewidth=1.5, alpha=0.6, label=f'Spot S={S_base}')
    ax.axhline(y=0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Prix Spot ($)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)

# Tableau récapitulatif dans la 6e case
ax6 = axes[5]
ax6.axis('off')

bs_ref = BlackScholes(S_base, K_base, T_base, r_base, sigma_base, 'call')
table_data = [
    ['Prix BS', f'${bs_ref.price():.4f}'],
    ['Delta Δ', f'{bs_ref.delta():.4f}'],
    ['Gamma Γ', f'{bs_ref.gamma():.6f}'],
    ['Vega ν',  f'{bs_ref.vega():.4f}'],
    ['Theta Θ', f'{bs_ref.theta():.4f}/day'],
    ['Rho ρ',   f'{bs_ref.rho():.4f}'],
]
table = ax6.table(
    cellText=table_data,
    colLabels=['Greek', f'Call (S={S_base}, K={K_base})'],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2)
ax6.set_title('Greeks au point de référence', fontsize=13, fontweight='bold')

plt.suptitle('Option Greeks — Analyse Complète | Black-Scholes Model', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('option_greeks.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée : option_greeks.png')

## 🌐 Partie 4 — Surface de Volatilité Implicite (Données Réelles)

La **volatilité implicite** est la volatilité que le marché "price" dans une option. La surface IV révèle le **smile** et le **skew** de volatilité.

In [ ]:
# ============================================================
# CELL 5 — DONNÉES RÉELLES YAHOO FINANCE + VOLATILITÉ IMPLICITE
# ============================================================

def implied_volatility(market_price, S, K, T, r, option_type='call', tol=1e-6):
    """
    Calcule la volatilité implicite par inversion numérique de Black-Scholes.
    Utilise la méthode de Brent (rapide et robuste).
    """
    if market_price <= 0 or T <= 0:
        return np.nan

    def objective(sigma):
        bs = BlackScholes(S, K, T, r, sigma, option_type)
        return bs.price() - market_price

    try:
        # Vérifier que la fonction change de signe
        if objective(0.001) * objective(5.0) >= 0:
            return np.nan
        iv = brentq(objective, 0.001, 5.0, xtol=tol)
        return iv if 0 < iv < 5 else np.nan
    except Exception:
        return np.nan


# ── Téléchargement des données Apple ─────────────────────────
print('📡 Téléchargement des données Apple (AAPL)...')
ticker = yf.Ticker('AAPL')

# Prix spot actuel
hist = ticker.history(period='1d')
S_real = hist['Close'].iloc[-1]
r_real = 0.053  # Fed funds rate approximatif

print(f'✅ AAPL Spot Price: ${S_real:.2f}')
print(f'📊 Taux sans risque: {r_real:.1%}')

# ── Récupération de la chaîne d'options ──────────────────────
expirations = ticker.options
print(f'\n📅 Expirations disponibles: {len(expirations)}')
print(f'  Prochaines: {expirations[:5]}')

In [ ]:
# ============================================================
# CELL 6 — CONSTRUCTION DE LA SURFACE DE VOLATILITÉ IMPLICITE
# ============================================================

import datetime

iv_data = []
today = datetime.datetime.today()

# Prendre les 6 premières expirations pour la surface
selected_expirations = expirations[:8]
print('🔄 Calcul des volatilités implicites...')

for exp_str in selected_expirations:
    try:
        exp_date = datetime.datetime.strptime(exp_str, '%Y-%m-%d')
        T = max((exp_date - today).days / 365.0, 0.01)

        chain = ticker.option_chain(exp_str)
        calls = chain.calls

        # Filtrer par liquidité et moneyness raisonnable
        calls = calls[
            (calls['volume'] > 10) &
            (calls['bid'] > 0.1) &
            (calls['strike'] >= S_real * 0.80) &
            (calls['strike'] <= S_real * 1.20)
        ]

        for _, row in calls.iterrows():
            K = row['strike']
            mid_price = (row['bid'] + row['ask']) / 2
            moneyness = K / S_real

            iv = implied_volatility(mid_price, S_real, K, T, r_real, 'call')

            if not np.isnan(iv) and 0.05 < iv < 2.0:
                iv_data.append({
                    'expiration': exp_str,
                    'T_years': T,
                    'T_days': int(T * 365),
                    'strike': K,
                    'moneyness': moneyness,
                    'log_moneyness': np.log(K / S_real),
                    'mid_price': mid_price,
                    'iv': iv,
                    'iv_pct': iv * 100,
                    'volume': row['volume'],
                })

    except Exception as e:
        print(f'  ⚠️  {exp_str}: {str(e)[:50]}')

df_iv = pd.DataFrame(iv_data)
print(f'\n✅ {len(df_iv)} points de volatilité calculés')
print(f'  Expirations: {df_iv["T_days"].unique()}')
print(f'  IV range: {df_iv["iv_pct"].min():.1f}% — {df_iv["iv_pct"].max():.1f}%')
df_iv.head(8)

In [ ]:
# ============================================================
# CELL 7 — VISUALISATION : SMILE + SURFACE 3D
# ============================================================

fig = plt.figure(figsize=(18, 7))

# ── Plot 1 : Volatility Smile par expiration ─────────────────
ax1 = fig.add_subplot(1, 2, 1)
colors_smile = plt.cm.viridis(np.linspace(0, 1, len(df_iv['expiration'].unique())))

for idx, (exp, group) in enumerate(df_iv.groupby('expiration')):
    group_sorted = group.sort_values('moneyness')
    ax1.plot(group_sorted['moneyness'], group_sorted['iv_pct'],
             marker='o', markersize=4, linewidth=2,
             color=colors_smile[idx], label=f'{exp} ({group["T_days"].iloc[0]}d)',
             alpha=0.85)

ax1.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='ATM (K=S)')
ax1.set_xlabel('Moneyness (K/S)', fontsize=12)
ax1.set_ylabel('Volatilité Implicite (%)', fontsize=12)
ax1.set_title(f'Volatility Smile — AAPL (S=${S_real:.1f})\nPar date d\'expiration', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right')

# ── Plot 2 : Term Structure de la Vol ATM ────────────────────
ax2 = fig.add_subplot(1, 2, 2)

# Vol ATM : closest to moneyness = 1.0
atm_vols = []
for exp, group in df_iv.groupby('expiration'):
    idx_atm = (group['moneyness'] - 1.0).abs().idxmin()
    atm_vols.append({
        'expiration': exp,
        'T_days': group.loc[idx_atm, 'T_days'],
        'atm_iv': group.loc[idx_atm, 'iv_pct']
    })
df_atm = pd.DataFrame(atm_vols).sort_values('T_days')

ax2.plot(df_atm['T_days'], df_atm['atm_iv'], 'o-', color='#1565C0', linewidth=2.5, markersize=8)
ax2.fill_between(df_atm['T_days'], df_atm['atm_iv'] * 0.95, df_atm['atm_iv'] * 1.05,
                  alpha=0.15, color='#1565C0')
ax2.set_xlabel('Jours avant expiration', fontsize=12)
ax2.set_ylabel('Volatilité Implicite ATM (%)', fontsize=12)
ax2.set_title('Term Structure — Volatilité ATM\nContango vs Backwardation', fontsize=13, fontweight='bold')

plt.suptitle('Implied Volatility Analysis — AAPL Options Chain', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('vol_smile.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée : vol_smile.png')

In [ ]:
# ============================================================
# CELL 8 — SURFACE 3D INTERACTIVE (PLOTLY)
# ============================================================

from scipy.interpolate import griddata

# Grille pour interpolation
moneyness_vals = df_iv['moneyness'].values
T_vals         = df_iv['T_days'].values
iv_vals        = df_iv['iv_pct'].values

# Créer une grille régulière
m_grid = np.linspace(moneyness_vals.min(), moneyness_vals.max(), 40)
T_grid = np.linspace(T_vals.min(), T_vals.max(), 40)
M_mesh, T_mesh = np.meshgrid(m_grid, T_grid)

# Interpolation bilinéaire
IV_surface = griddata(
    (moneyness_vals, T_vals),
    iv_vals,
    (M_mesh, T_mesh),
    method='linear'
)

# Surface 3D Plotly
fig_3d = go.Figure()

fig_3d.add_trace(go.Surface(
    x=M_mesh, y=T_mesh, z=IV_surface,
    colorscale='Viridis',
    colorbar=dict(title='IV (%)', tickfont=dict(size=12)),
    opacity=0.9,
    name='Vol Surface'
))

# Points de données réels
fig_3d.add_trace(go.Scatter3d(
    x=moneyness_vals, y=T_vals, z=iv_vals,
    mode='markers',
    marker=dict(size=3, color='white', opacity=0.7),
    name='Market data'
))

fig_3d.update_layout(
    title=dict(
        text=f'<b>Implied Volatility Surface — AAPL</b><br><sub>Spot: ${S_real:.1f} | Données marché en temps réel</sub>',
        x=0.5, font=dict(size=16)
    ),
    scene=dict(
        xaxis_title='Moneyness (K/S)',
        yaxis_title='Jours avant expiration',
        zaxis_title='Volatilité Implicite (%)',
        xaxis=dict(backgroundcolor='rgb(240,240,250)'),
        camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2))
    ),
    width=850, height=600
)

fig_3d.show()
fig_3d.write_html('vol_surface_3d.html')
print('✅ Surface 3D interactive sauvegardée : vol_surface_3d.html')

## 💰 Partie 5 — P&L d'une Stratégie Options (Risk Reversal)

In [ ]:
# ============================================================
# CELL 9 — STRATÉGIES OPTIONS ET P&L
# ============================================================

def option_payoff(S_range, K, premium, option_type='call', position='long'):
    """Calcule le P&L d'une option à expiration"""
    if option_type == 'call':
        intrinsic = np.maximum(S_range - K, 0)
    else:
        intrinsic = np.maximum(K - S_range, 0)

    pnl = intrinsic - premium
    if position == 'short':
        pnl = -pnl
    return pnl


S_plot = np.linspace(140, 250, 500)
S0 = 190

# ── Stratégies à visualiser ───────────────────────────────────
strategies = {
    'Long Call (K=195)': {
        'legs': [(option_payoff(S_plot, 195, 5.20, 'call', 'long'), '#1565C0', '-', 2.5)],
    },
    'Bull Call Spread\n(Long 190C, Short 200C)': {
        'legs': [
            (option_payoff(S_plot, 190, 8.50, 'call', 'long'),  '#1565C0', '--', 1.5),
            (option_payoff(S_plot, 200, 3.80, 'call', 'short'), '#C62828', '--', 1.5),
        ]
    },
    'Long Straddle\n(Long 190C + Long 190P)': {
        'legs': [
            (option_payoff(S_plot, 190, 8.50, 'call', 'long'), '#1565C0', '--', 1.5),
            (option_payoff(S_plot, 190, 7.20, 'put',  'long'), '#7B1FA2', '--', 1.5),
        ]
    },
    'Risk Reversal\n(Long 200C, Short 180P)': {
        'legs': [
            (option_payoff(S_plot, 200, 3.80, 'call', 'long'),  '#1565C0', '--', 1.5),
            (option_payoff(S_plot, 180, 4.10, 'put',  'short'), '#C62828', '--', 1.5),
        ]
    }
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (strat_name, strat_info) in enumerate(strategies.items()):
    ax = axes[idx]

    # P&L total de la stratégie
    total_pnl = sum(leg[0] for leg in strat_info['legs'])

    # Afficher chaque leg
    for leg_pnl, color, ls, lw in strat_info['legs']:
        ax.plot(S_plot, leg_pnl, color=color, linestyle=ls, linewidth=lw, alpha=0.5)

    # P&L total
    ax.plot(S_plot, total_pnl, color='black', linewidth=3, label='P&L net')

    # Zone profits/pertes
    ax.fill_between(S_plot, total_pnl, 0,
                    where=total_pnl >= 0, alpha=0.15, color='green', label='Profit')
    ax.fill_between(S_plot, total_pnl, 0,
                    where=total_pnl < 0,  alpha=0.15, color='red',   label='Perte')

    ax.axhline(y=0,  color='black', linewidth=1.0)
    ax.axvline(x=S0, color='gray',  linewidth=1.5, linestyle=':', label=f'Spot S={S0}')

    # Breakeven
    sign_changes = np.where(np.diff(np.sign(total_pnl)))[0]
    for sc in sign_changes:
        be_price = S_plot[sc]
        ax.axvline(x=be_price, color='orange', linewidth=1.5, linestyle='--', alpha=0.8)
        ax.annotate(f'BE=${be_price:.0f}', xy=(be_price, 0),
                    xytext=(be_price + 2, max(total_pnl) * 0.3),
                    fontsize=9, color='darkorange')

    ax.set_xlabel('Prix spot à expiration ($)', fontsize=11)
    ax.set_ylabel('Profit / Perte ($)', fontsize=11)
    ax.set_title(strat_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)

plt.suptitle('Options Strategies — P&L Analysis at Expiration', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('options_strategies_pnl.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée : options_strategies_pnl.png')

## 📊 Partie 6 — Convergence Monte Carlo & Analyse de Sensibilité

In [ ]:
# ============================================================
# CELL 10 — CONVERGENCE MC + SENSITIVITY ANALYSIS
# ============================================================

# ── 1. Convergence Monte Carlo ────────────────────────────────
n_sims_list = [100, 500, 1000, 5000, 10000, 50000, 100000]
bs_ref_price = BlackScholes(S, K, T, r, sigma, 'call').price()

mc_prices = []
mc_errors = []
for n in n_sims_list:
    price, se, _ = monte_carlo_pricer(S, K, T, r, sigma, 'call', n_simulations=n, n_steps=50)
    mc_prices.append(price)
    mc_errors.append(1.96 * se)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
ax1.semilogx(n_sims_list, mc_prices, 'o-', color='#1565C0', linewidth=2.5, markersize=7, label='MC Price')
ax1.fill_between(n_sims_list,
                  [p - e for p, e in zip(mc_prices, mc_errors)],
                  [p + e for p, e in zip(mc_prices, mc_errors)],
                  alpha=0.2, color='#1565C0', label='95% CI')
ax1.axhline(y=bs_ref_price, color='#C62828', linewidth=2.5, linestyle='--', label=f'BS Price = ${bs_ref_price:.4f}')
ax1.set_xlabel('Nombre de simulations (log scale)', fontsize=12)
ax1.set_ylabel('Prix estimé ($)', fontsize=12)
ax1.set_title('Convergence Monte Carlo vs Black-Scholes', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

# ── 2. Sensitivity Analysis (Heatmap Delta-Vol) ───────────────
ax2 = axes[1]
S_heat  = np.linspace(160, 220, 20)
vol_heat = np.linspace(0.10, 0.60, 20)

price_matrix = np.array([
    [BlackScholes(s, K, T, r, v, 'call').price() for s in S_heat]
    for v in vol_heat
])

im = ax2.contourf(S_heat, vol_heat * 100, price_matrix, levels=20, cmap='RdYlGn')
plt.colorbar(im, ax=ax2, label='Option Price ($)')
ax2.contour(S_heat, vol_heat * 100, price_matrix, levels=10, colors='white', linewidths=0.5, alpha=0.5)
ax2.axvline(x=K, color='white', linewidth=2, linestyle='--', label=f'Strike K={K}')
ax2.set_xlabel('Prix Spot ($)', fontsize=12)
ax2.set_ylabel('Volatilité (%)', fontsize=12)
ax2.set_title('Heatmap : Prix Call vs Spot & Volatilité', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)

plt.suptitle('Monte Carlo Convergence & Sensitivity Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('convergence_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure sauvegardée : convergence_sensitivity.png')

## 📋 Partie 7 — Rapport de Synthèse Final

In [ ]:
# ============================================================
# CELL 11 — RAPPORT DE SYNTHÈSE PROFESSIONNEL
# ============================================================

print('═' * 65)
print('       OPTIONS PRICING ENGINE — RAPPORT FINAL')
print('═' * 65)
print(f'  Sous-jacent    : AAPL (Apple Inc.)')
print(f'  Spot price     : ${S_real:.2f}')
print(f'  Risk-free rate : {r_real:.1%} (Fed Funds Rate)')
print('─' * 65)
print('  RÉSULTATS DE PRICING')
print('─' * 65)

# Plusieurs strikes et maturités
strikes_test = [S_real * 0.95, S_real, S_real * 1.05]
maturities   = [30/365, 60/365, 90/365]
vol_atm      = df_iv[df_iv['moneyness'].between(0.99, 1.01)]['iv'].mean() if len(df_iv) > 0 else 0.28

print(f"{'Strike':>10} {'Matu':>8} {'BS Call':>10} {'BS Put':>10} {'Delta':>8} {'Vega':>8}")
print('─' * 65)
for T_t in maturities:
    for K_t in strikes_test:
        bs_c = BlackScholes(S_real, K_t, T_t, r_real, vol_atm, 'call')
        bs_p = BlackScholes(S_real, K_t, T_t, r_real, vol_atm, 'put')
        print(f"  ${K_t:>7.2f}  {int(T_t*365):>5}d  ${bs_c.price():>8.3f}  ${bs_p.price():>8.3f}  {bs_c.delta():>7.3f}  {bs_c.vega():>7.4f}")

print('─' * 65)
print(f'  Vol implicite ATM utilisée : {vol_atm:.1%}')
print('─' * 65)
print('  OUTPUTS DU PROJET')
print('─' * 65)
print('  ✅ monte_carlo_simulation.png  — Trajectoires & distribution')
print('  ✅ option_greeks.png           — Analyse complète des Greeks')
print('  ✅ vol_smile.png               — Volatility smile & term structure')
print('  ✅ vol_surface_3d.html         — Surface IV 3D interactive')
print('  ✅ options_strategies_pnl.png  — P&L 4 stratégies')
print('  ✅ convergence_sensitivity.png — Convergence MC & heatmap')
print('═' * 65)
print()
print('  SKILLS DÉMONTRÉS (CV) :')
print('  • Stochastic calculus & Black-Scholes model')
print('  • Monte Carlo simulation (antithetic variates)')
print('  • Implied volatility extraction (Brent method)')
print('  • Volatility surface modelling')
print('  • Option Greeks computation & interpretation')
print('  • Real market data analysis (Yahoo Finance API)')
print('  • Options strategies P&L analysis')
print('═' * 65)

---
## 🚀 Pour aller plus loin

1. **Modèle de Heston** — stochastic volatility pour mieux capturer le smile
2. **Options américaines** — algorithme de Longstaff-Schwartz pour l'exercice anticipé
3. **Couverture dynamique Delta-Gamma** — simulation d'une stratégie de hedging
4. **SVI parametrization** — fit analytique de la surface de volatilité

---
*Projet réalisé dans le cadre d'une recherche quantitative personnelle.*  
*Données : Yahoo Finance | Modèle : Black-Scholes (1973)*